In [1]:
import numpy as np
import skysurvey
import pandas
import scipy 
import sncosmo
import astropy

This notebook is based on the `skysurvey` library. Documentation is available at https://skysurvey.readthedocs.io/.

### 1. Simulate SNe Ia with the base skysurvey model

In [2]:
# Default model in skysurvey
skysurvey.SNeIa().model

{'redshift': {'func': 'draw_redshift', 'kwargs': {'zmax': 0.2}, 'as': 'z'},
 'x1': {'func': <function SNeIaStretch.nicolas2021 at 0x31c02ecf0>,
        'kwargs': {}},
 'c': {'func': <function SNeIaColor.intrinsic_and_dust at 0x31c02ec40>,
       'kwargs': {}},
 't0': {'func': <function random_uniform at 0x31bf9fcc0>,
        'kwargs': {'low': 56000, 'high': 56200}},
 'magabs': {'func': <function SNeIaMagnitude.tripp1998 at 0x31c02eda0>,
            'kwargs': {'x1': '@x1', 'c': '@c', 'mabs': -19.3, 'sigmaint': 0.1}},
 'magobs': {'func': 'magabs_to_magobs',
            'kwargs': {'z': '@z', 'magabs': '@magabs'}},
 'radec': {'func': <function random_radec at 0x31bfe1900>,
           'kwargs': {},
           'as': ['ra', 'dec']}}

In [3]:
snia = skysurvey.SNeIa()

To change the number of SNe calculated, you can:
- Provide the number of SNe, the redshift range and the time. The rate here is used to compute the shape of the redshift histogram but it is scaled to match the number of SNe required.
- Provide a redshift range and a time range. The rate is used to compute the number of SNe.

In [4]:
data = snia.draw(zmin=0, zmax=0.15, tstart=0, tstop=365)

In [5]:
data

,z,x1,c,t0,ra,dec,magabs,magobs
0,0.11215,0.565,-0.006,310.823366,149.878681,34.863587,-19.279698,19.373846
1,0.11745,0.610,0.273,19.901519,216.205706,-63.640416,-18.542157,20.219119
2,0.14275,-1.230,-0.054,101.302140,318.122797,4.337442,-19.101158,20.118602
3,0.13225,-0.435,0.041,210.503492,48.770092,21.084018,-19.091230,19.948307
4,0.13405,0.330,0.091,84.173009,73.245980,63.932927,-19.030670,20.040693
...,...,...,...,...,...,...,...,...
21033,0.10255,0.790,-0.087,178.609116,109.255046,-17.500787,-19.641343,18.804220
21034,0.10315,-1.730,-0.017,145.171251,175.668107,-55.068514,-19.124380,19.334710
21035,0.12885,0.120,0.589,47.539843,69.606861,-61.482713,-17.533396,21.444902
21036,0.14945,1.295,-0.022,12.732686,352.390663,38.730061,-19.542684,19.785703


In [6]:
snia.set_data(data)

### 2. Complexify the model based on what we saw in the lectures

You can modify the parameters of the distributions, change the distribution shapes, add extra dependencies, change the standardisation...

In [ ]:
def tripp98_standardisation(x1, c):
    return 

In [ ]:
SNeIa = dict(redshift = {"kwargs": {"zmax":0.2}, "as":"z"}, # This cannot be modified as it's drawn outside the DAG following a rate
                   
                    x1 = {"func": scipy.stats.norm.rvs, 
                        "kwargs": { "loc":0, "scale":1}}, 
                   
                   c = {"func": scipy.stats.norm.rvs,
                       "kwargs": {"loc":0, 
                                  "scale":0.1}},

                   t0 = {"func": np.random.uniform, 
                         "kwargs": {"low":56_000, "high":56_200} },
                       
                   magabs = {"func": tripp98_standardisation, # the @ points towards another parameter in the graph
                             "kwargs": {"x1": "@x1", "c": "@c",
                                        "mabs":-19.3,
                                        "sigmaint":0.1, 
                                        "alpha":-0.15, 
                                        "beta":3.15}},

                    magobs = {"func": "magabs_to_magobs", # str -> method of the class
                             "kwargs": {"z":"@z", "magabs":"@magabs"},},
                       
                   radec = {"func": skysurvey.tools.utils.random_radec,
                            "kwargs": {},
                            "as": ["ra","dec"]})

In [ ]:
snia.set_model(SNeIa)

You can also change the sncosmo template (e.g. SALT2, SALT3), the cosmology (it needs to be an astropy cosmology), the rate (either a constant or a function of z), and add effects such as MW extinction.

In [8]:
snia.set_template(sncosmo.Model(source=sncosmo.get_source('salt3')))
snia.set_cosmology(astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.315))
snia.set_rate(23500) # Default Perley+20 rate
snia.add_effect(skysurvey.effects.mw_extinction) # Change that

### 3a. Simulate the LCs from ZTF logs

In [13]:
# Load the ZTF logs 

ztf = skysurvey.ZTF.from_logs()
print(f"{len(ztf.data):,} quadrant observations")
ztf.data.head()
error_scale = {"ztfg": 1.23, "ztfr": 1.17, "ztfi": 1.20}
ztf.data["skynoise"] = ztf.data["skynoise"] * ztf.data["band"].map(error_scale).astype("float32")

32,808,495 quadrant observations


In [ ]:
# Here simulate SNe following a model

In [ ]:
dset = skysurvey.DataSet.from_targets_and_survey(snia, ztf, incl_error=True, phase_range=[-20, 40])

### 3b. "Noisify" the sample directly from the true parameters

In [42]:
# Here is an example on how to apply noise to magobs
errormodel = {"magobs": {"func": scipy.stats.norm.rvs, "kwargs": {"loc": 0.1, "scale": 0.01}}}

noisy_snia = snia.apply_gaussian_noise(errormodel)
noisy_snia.data[["magobs_true", "magobs_err", "magobs"]]

,magobs_true,magobs_err,magobs
0,20.919978,0.098308,20.883518
1,19.814004,0.101529,19.809043
2,20.093286,0.105590,19.790380
3,20.320581,0.095639,20.248992
4,19.000331,0.111390,19.059603
...,...,...,...
21033,20.854345,0.115474,20.821398
21034,19.042388,0.089433,19.048410
21035,19.513113,0.094648,19.556921
21036,19.779182,0.091586,19.541466


### 4. Apply selection so that the samples match the data

Extra:
- Fit the LCs from 3b with sncosmo
- Fit standardisation on your sample from 3a or 3b